In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches

In [3]:
import pandas as pd

data = [
    # ---------------- DATA / AI ----------------
    {
        "job_role": "Data Scientist",
        "domain": "Data",
        "level": "Mid",
        "skills": ["python", "sql", "machine learning", "statistics", "data visualization"],
        "certifications": ["AWS Data Analytics", "TensorFlow Developer"],
        "next_roles": ["Machine Learning Engineer", "AI Specialist"]
    },
    {
        "job_role": "Data Analyst",
        "domain": "Data",
        "level": "Junior",
        "skills": ["excel", "sql", "python", "data visualization"],
        "certifications": ["Google Data Analytics"],
        "next_roles": ["Data Scientist", "Business Analyst"]
    },
    {
        "job_role": "Data Engineer",
        "domain": "Data",
        "level": "Mid",
        "skills": ["python", "sql", "spark", "etl", "airflow"],
        "certifications": ["AWS Data Engineer"],
        "next_roles": ["Senior Data Engineer", "Data Architect"]
    },
    {
        "job_role": "Business Analyst",
        "domain": "Data",
        "level": "Junior",
        "skills": ["excel", "sql", "powerbi", "communication"],
        "certifications": ["CBAP"],
        "next_roles": ["Data Analyst", "Product Manager"]
    },
    {
        "job_role": "Machine Learning Engineer",
        "domain": "AI",
        "level": "Mid",
        "skills": ["python", "machine learning", "tensorflow", "pytorch", "deep learning"],
        "certifications": ["AWS ML Specialty"],
        "next_roles": ["AI Engineer", "Research Scientist"]
    },

    # ---------------- AI / ADVANCED AI ----------------
    {
        "job_role": "AI Engineer",
        "domain": "AI",
        "level": "Senior",
        "skills": ["python", "machine learning", "deep learning", "nlp", "llms"],
        "certifications": ["Deep Learning Specialization"],
        "next_roles": ["AI Architect"]
    },
    {
        "job_role": "NLP Engineer",
        "domain": "AI",
        "level": "Mid",
        "skills": ["python", "nlp", "transformers", "machine learning"],
        "certifications": ["HuggingFace NLP Certificate"],
        "next_roles": ["AI Engineer"]
    },
    {
        "job_role": "Computer Vision Engineer",
        "domain": "AI",
        "level": "Mid",
        "skills": ["python", "opencv", "deep learning", "cnn"],
        "certifications": ["Deep Learning Specialization"],
        "next_roles": ["AI Engineer"]
    },

    # ---------------- WEB DEVELOPMENT ----------------
    {
        "job_role": "Frontend Developer",
        "domain": "Web",
        "level": "Junior",
        "skills": ["html", "css", "javascript", "react"],
        "certifications": ["Meta Frontend Certificate"],
        "next_roles": ["Full Stack Developer"]
    },
    {
        "job_role": "Backend Developer",
        "domain": "Web",
        "level": "Junior",
        "skills": ["python", "django", "sql", "apis", "git"],
        "certifications": ["AWS Developer Associate"],
        "next_roles": ["Full Stack Developer"]
    },
    {
        "job_role": "Full Stack Developer",
        "domain": "Web",
        "level": "Mid",
        "skills": ["html", "css", "javascript", "react", "nodejs", "python"],
        "certifications": ["Full Stack Certification"],
        "next_roles": ["Senior Software Engineer"]
    },
    {
        "job_role": "Mobile App Developer",
        "domain": "Web",
        "level": "Mid",
        "skills": ["flutter", "dart", "firebase", "apis"],
        "certifications": ["Google Flutter Certificate"],
        "next_roles": ["Senior Mobile Developer"]
    },

    # ---------------- DEVOPS / CLOUD ----------------
    {
        "job_role": "DevOps Engineer",
        "domain": "DevOps",
        "level": "Mid",
        "skills": ["linux", "docker", "kubernetes", "aws", "ci/cd"],
        "certifications": ["AWS DevOps Engineer"],
        "next_roles": ["Cloud Architect"]
    },
    {
        "job_role": "Cloud Engineer",
        "domain": "DevOps",
        "level": "Mid",
        "skills": ["aws", "azure", "cloud computing", "linux"],
        "certifications": ["AWS Solutions Architect"],
        "next_roles": ["Cloud Architect"]
    },
    {
        "job_role": "Site Reliability Engineer",
        "domain": "DevOps",
        "level": "Senior",
        "skills": ["linux", "monitoring", "docker", "kubernetes", "python"],
        "certifications": ["Google SRE Certificate"],
        "next_roles": ["DevOps Lead"]
    },

    # ---------------- SOFTWARE ENGINEERING ----------------
    {
        "job_role": "Software Engineer",
        "domain": "Software",
        "level": "Junior",
        "skills": ["python", "java", "git", "algorithms"],
        "certifications": ["Oracle Java Certificate"],
        "next_roles": ["Senior Software Engineer"]
    },
    {
        "job_role": "System Engineer",
        "domain": "Software",
        "level": "Mid",
        "skills": ["linux", "networking", "python", "system design"],
        "certifications": ["CompTIA Linux+"],
        "next_roles": ["System Architect"]
    },

    # ---------------- EXTENDED (to reach 55+) ----------------
]

# 🔥 Auto-expand dataset to 55+ roles
base_roles = [
    ("QA Engineer", "Software", "Junior", ["testing", "selenium", "automation", "python"]),
    ("Automation Engineer", "Software", "Mid", ["python", "automation", "ci/cd", "testing"]),
    ("Database Administrator", "Data", "Mid", ["sql", "database", "optimization", "backup"]),
    ("Data Architect", "Data", "Senior", ["sql", "system design", "data modeling", "cloud"]),
    ("AI Research Scientist", "AI", "Senior", ["python", "research", "deep learning", "math"]),
    ("Blockchain Developer", "Software", "Mid", ["blockchain", "solidity", "cryptography"]),
    ("Game Developer", "Software", "Mid", ["unity", "c#", "game design"]),
    ("Security Engineer", "Cybersecurity", "Mid", ["networking", "security", "linux", "encryption"]),
    ("Ethical Hacker", "Cybersecurity", "Mid", ["penetration testing", "linux", "networking"]),
    ("UI/UX Designer", "Design", "Junior", ["figma", "ui design", "ux research"]),
    ("Product Manager", "Business", "Senior", ["communication", "agile", "roadmapping"]),
    ("Scrum Master", "Business", "Mid", ["agile", "scrum", "team management"]),
    ("Technical Writer", "Software", "Junior", ["writing", "documentation", "communication"]),
    ("AI Product Manager", "AI", "Senior", ["ai", "product management", "machine learning"]),
    ("Data Visualization Engineer", "Data", "Mid", ["tableau", "powerbi", "python", "visualization"]),
]

for role, domain, level, skills in base_roles:
    data.append({
        "job_role": role,
        "domain": domain,
        "level": level,
        "skills": skills,
        "certifications": [],
        "next_roles": []
    })

# Convert to DataFrame
df = pd.DataFrame(data)

# Ensure we hit 55+ rows by duplicating intelligently with variation if needed
while len(df) < 55:
    extra = df.sample(1).iloc[0].to_dict()
    extra["job_role"] = extra["job_role"] + " (Level " + str(len(df)) + ")"
    df = pd.concat([df, pd.DataFrame([extra])], ignore_index=True)

# Save CSV
df.to_csv("industry_jobs.csv", index=False)

print("Dataset created with rows:", len(df))
df.head()

Dataset created with rows: 55


,job_role,domain,level,skills,certifications,next_roles
0,Data Scientist,Data,Mid,"[python, sql, machine learning, statistics, da...","[AWS Data Analytics, TensorFlow Developer]","[Machine Learning Engineer, AI Specialist]"
1,Data Analyst,Data,Junior,"[excel, sql, python, data visualization]",[Google Data Analytics],"[Data Scientist, Business Analyst]"
2,Data Engineer,Data,Mid,"[python, sql, spark, etl, airflow]",[AWS Data Engineer],"[Senior Data Engineer, Data Architect]"
3,Business Analyst,Data,Junior,"[excel, sql, powerbi, communication]",[CBAP],"[Data Analyst, Product Manager]"
4,Machine Learning Engineer,AI,Mid,"[python, machine learning, tensorflow, pytorch...",[AWS ML Specialty],"[AI Engineer, Research Scientist]"


In [9]:
# Convert all skill columns to lowercase
df['skills'] = df['skills'].apply(lambda x: str(x).lower())

In [10]:
# Convert all skill columns to lowercase
df['skills'] = df['skills'].apply(lambda x: str(x).lower())

In [11]:
import ast

df['skills'] = df['skills'].apply(lambda x: ast.literal_eval(x))
df['skills'] = df['skills'].apply(lambda skills: [s.lower().strip() for s in skills])

In [12]:
master_skills = sorted(list(set(
    skill for skills in df['skills'] for skill in skills
)))

In [13]:
def fuzzy_match(skill, master_list):
    match = get_close_matches(skill, master_list, n=1, cutoff=0.7)
    return match[0] if match else skill

In [14]:
def clean_user_skills(user_skills):
    cleaned = []
    for skill in user_skills:
        skill = skill.lower().strip()
        skill = fuzzy_match(skill, master_skills)
        cleaned.append(skill)
    return cleaned

In [16]:
skill_weights = {}

for skill in master_skills:
    skill_weights[skill] = 1  # default weight

# You can manually boost important skills
important_skills = ['python', 'machine learning', 'sql']

for skill in important_skills:
    if skill in skill_weights:
        skill_weights[skill] = 3

In [17]:
def vectorize(skills):
    return np.array([
        skill_weights.get(skill, 1) if skill in skills else 0
        for skill in master_skills
    ])

In [18]:
df['vector'] = df['skills'].apply(vectorize)

In [19]:
user_input = input("Enter your skills (comma separated): ")
user_skills = user_input.split(',')

user_skills = clean_user_skills(user_skills)
user_vector = vectorize(user_skills)

print("Cleaned Skills:", user_skills)

Enter your skills (comma separated):  python,ml ,c++,java


Cleaned Skills: ['python', 'ml', 'c++', 'java']


In [20]:
job_vectors = np.stack(df['vector'].values)

scores = cosine_similarity([user_vector], job_vectors)[0]

df['score'] = scores

In [21]:
top_n = 5

recommendations = df.sort_values(by='score', ascending=False).head(top_n)

recommendations[['job_role', 'score']]

,job_role,score
15,Software Engineer,0.912871
53,Software Engineer (Level 53),0.912871
21,AI Research Scientist,0.821584
17,QA Engineer,0.821584
18,Automation Engineer,0.821584


In [22]:
def explain(row_skills, user_skills):
    matched = list(set(row_skills).intersection(set(user_skills)))
    missing = list(set(row_skills) - set(user_skills))
    return matched, missing

In [23]:
for _, row in recommendations.iterrows():
    matched, missing = explain(row['skills'], user_skills)
    
    print("\n======================")
    print("Job Role:", row['job_role'])
    print("Score:", round(row['score'], 3))
    print("Matched Skills:", matched)
    print("Missing Skills:", missing)


Job Role: Software Engineer
Score: 0.913
Matched Skills: ['python', 'java']
Missing Skills: ['algorithms', 'git']

Job Role: Software Engineer (Level 53)
Score: 0.913
Matched Skills: ['python', 'java']
Missing Skills: ['algorithms', 'git']

Job Role: AI Research Scientist
Score: 0.822
Matched Skills: ['python']
Missing Skills: ['research', 'math', 'deep learning']

Job Role: QA Engineer
Score: 0.822
Matched Skills: ['python']
Missing Skills: ['selenium', 'automation', 'testing']

Job Role: Automation Engineer
Score: 0.822
Matched Skills: ['python']
Missing Skills: ['ci/cd', 'automation', 'testing']
